In [4]:
import pandas as pd
import os
filepath = "../DATA-HTML-STOCK/NEPSECompany/NEPSECompanyExtractor.csv"
symbols = pd.read_csv(filepath)['Symbol'].tolist()

stock_ignored = ["CORBLP", "GFCLPO", "HBLPO", "LFCPO", "SBIPO", "GABLPO", "AMFIPO", "NMLBS", "ILFCPO"]
stock_no_news = ["HATHPO", "BUDBLP", "CFCLPO", "CORBLP", "EBLPO", "HAMROP", "HGIPO", "JFLPO", "JBNLPO", "KADBLP", "KNBLPO", "CEFLPO", "LBLPO", "MFILPO", "MIDBLP", "NBBLPO", "NABBPO", "NBBPO", "PFLPO", "PRINPO", "PURBLP", "SBBLJP", "SIFCPO", "SILPO", "SMFDBP", "SYFLPO", "TNBLPO", "TDBLPO", "UFLPO", "WDBLPO", "WMBFPO", "HLBSLP"]

def detailed_prediction(score):
    if score >= 0.60: return "Strongly Positive"
    elif score >= 0.20: return "Positive"
    elif score >= 0.05: return "Slightly Positive"
    elif score > -0.05: return "Neutral"
    elif score > -0.20: return "Slightly Negative"
    elif score > -0.60: return "Negative"
    else: return "Strongly Negative"

def check_existing_semifinal(savepath):
    if os.path.exists(savepath):
        old_df = pd.read_csv(savepath)
        old_df["Date_Stock"] = pd.to_datetime(old_df["Date_Stock"])
        return old_df["Date_Stock"].max(), old_df
    return None, None


for symbol in symbols:

    print(f"\nWorking on {symbol}")

    if symbol in stock_ignored:
        print(f"{symbol} ignored")
        continue

    stockpath = f"../DATA-HTML-STOCK/NEPSEDATA/{symbol}.csv"
    newspath  = f"../DATA-HTML-STOCK/STOCKSENTIMENT/{symbol}news_sentiment.csv"
    savepath  = f"../DATA-HTML-STOCK/SemiFinalDataset/{symbol}.csv"

    if not os.path.exists(stockpath):
        print("No stock data")
        continue

    
    stock_df = pd.read_csv(stockpath)
    stock_df["Date"] = pd.to_datetime(stock_df["Date"])

    stock_df = stock_df[["Date", "Close"]].rename(columns={"Date": "Date_Stock"})
    stock_df = stock_df.sort_values("Date_Stock")

    saved_date, old_df = check_existing_semifinal(savepath)

    if saved_date is not None:
        stock_df = stock_df[stock_df["Date_Stock"] > saved_date]

    if stock_df.empty:
        print("No new stock data")
        continue

    
    if symbol in stock_no_news or not os.path.exists(newspath):
        news_df = pd.DataFrame(columns=["Date_News", "Sentiment_Score"])
    else:
        news_df = pd.read_csv(newspath)
        news_df["Date"] = pd.to_datetime(news_df["Date"])

        
        news_df = news_df.groupby("Date")["Sentiment_Score"].mean().reset_index()
        news_df = news_df.rename(columns={"Date": "Date_News"})

    
    new_df = pd.merge(
        stock_df,
        news_df,
        left_on="Date_Stock",
        right_on="Date_News",
        how="left"
    )

    
    new_df["Sentiment_Score"] = new_df["Sentiment_Score"].fillna(0)
    new_df["Prediction"] = new_df["Sentiment_Score"].apply(detailed_prediction)

    new_df = new_df[["Date_Stock", "Close", "Date_News", "Sentiment_Score", "Prediction"]]

    if old_df is not None:
        final_df = pd.concat([old_df, new_df], ignore_index=True)
    else:
        final_df = new_df

    final_df = final_df.drop_duplicates(subset=["Date_Stock"])
    final_df = final_df.sort_values("Date_Stock")

    final_df.to_csv(savepath, index=False)

    print(f"{symbol} done: {len(final_df)} rows")

print("All Done")


Working on ADBL
ADBL done: 3523 rows

Working on API
API done: 2356 rows

Working on HATH
HATH done: 716 rows

Working on HATHPO
HATHPO done: 1 rows

Working on AKPL
AKPL done: 2004 rows

Working on AHPC
AHPC done: 3616 rows

Working on ALICL
ALICL done: 3482 rows

Working on ALICLP
ALICLP done: 73 rows

Working on BOKL
BOKL done: 1321 rows

Working on BOKLPO
BOKLPO done: 42 rows

Working on BARUN
BARUN done: 2281 rows

Working on BFC
BFC done: 1788 rows

Working on BFCPO
BFCPO done: 28 rows

Working on BHBL
BHBL done: 877 rows

Working on BHBLPO
BHBLPO done: 9 rows

Working on BBC
BBC done: 2236 rows

Working on BNT
BNT done: 1908 rows

Working on BNL
BNL done: 392 rows

Working on BUDBL
BUDBL done: 600 rows

Working on BUDBLP
BUDBLP done: 4 rows

Working on BPCL
BPCL done: 3621 rows

Working on CFCL
CFCL done: 2802 rows

Working on CFCLPO
CFCLPO done: 4 rows

Working on CCBL
CCBL done: 1892 rows

Working on CCBLPO
CCBLPO done: 143 rows

Working on CBBL
CBBL done: 3113 rows

Working 